In [1]:
import tonic
import tonic.transforms as transforms  # Not to be mistaken with torchdata.transfroms
from tonic import DiskCachedDataset

# torch imports
import torch
from torch.utils.data import random_split
from torch.utils.data import DataLoader
import torchvision
import torch.nn as nn


# snntorch imports
import snntorch as snn
from snntorch import surrogate
import snntorch.spikeplot as splt
from snntorch import functional as SF
from snntorch import utils


# other imports
import matplotlib.pyplot as plt
from IPython.display import HTML
from IPython.display import display
import numpy as np
import torchdata
import os
from ipywidgets import IntProgress
import time
import statistics

root = "/home/emre/Downloads/"  # similar to os.path.join('content', 'drive', 'My Drive')
os.listdir(os.path.join(root, 'STMNIST')) # confirm the file exists


dataset = tonic.prototype.datasets.STMNIST(root=root, keep_compressed = False, shuffle = False)

events, target = next(iter(dataset))

sensor_size = tuple(tonic.prototype.datasets.STMNIST.sensor_size.values())  # The sensor size for STMNIST is (10, 10, 2)

# filter noisy pixels and integrate events into 1ms frames
frame_transform = transforms.Compose([transforms.Denoise(filter_time=10000),
                                      transforms.ToFrame(sensor_size=sensor_size,
                                                         time_window=20000)
                                     ])

transformed_events = frame_transform(events)



sensor_size = tonic.prototype.datasets.STMNIST.sensor_size
sensor_size = tuple(sensor_size.values())

# Define a transform
frame_transform = transforms.Compose([transforms.ToFrame(sensor_size=sensor_size, time_window=20000)])


def transform_STMNIST(data, transform):
    # total sample, 6953.
    train_size = 5562 # Around 80% for train
    val_size = 696 # Around 10% for validation 
    test_size = 695 # Around 10% for test

    train_bar = IntProgress(min=0, max=train_size)
    test_bar = IntProgress(min=0, max=test_size)
    val_bar = IntProgress(min=0, max=val_size)

    testset = []
    trainset = []
    valset = []

    print('Porting over and transforming the trainset.')
    display(train_bar)
    for _ in range(train_size):
        events, target = next(iter(dataset))
        events = transform(events)
        trainset.append((events, target))
        train_bar.value += 1

    print('Porting over and transforming the testset.')
    display(test_bar)
    for _ in range(test_size):
        events, target = next(iter(dataset))
        events = transform(events)
        testset.append((events, target))
        test_bar.value += 1

    print('Porting over and transforming the valset.')
    display(val_bar)
    for _ in range(val_size):
        events, target = next(iter(dataset))
        events = transform(events)
        valset.append((events, target))
        val_bar.value += 1

    return (trainset, testset, valset)

start_time = time.time()
trainset, testset, valset = transform_STMNIST(dataset, frame_transform)
elapsed_time = time.time() - start_time

# Convert elapsed time to minutes, seconds, and milliseconds
minutes, seconds = divmod(elapsed_time, 60)
seconds, milliseconds = divmod(seconds, 1)
milliseconds = round(milliseconds * 1000)

# Print the elapsed time
print(f"Elapsed time: {int(minutes)} minutes, {int(seconds)} seconds, {milliseconds} milliseconds")


# Create a DataLoader
dataloader = DataLoader(trainset, batch_size=32, shuffle=True)


transform = tonic.transforms.Compose([torch.from_numpy])

cached_trainset = DiskCachedDataset(trainset, transform=transform, cache_path='./cache/stmnist/train')

# no augmentations for the testset
cached_testset = DiskCachedDataset(testset, cache_path='./cache/stmnist/test')

cached_valset = DiskCachedDataset(valset, cache_path='./cache/stmnist/val')

print("Baking the cache to disk... (this creates the files)")
for _ in cached_trainset: pass
for _ in cached_testset: pass
for _ in cached_valset: pass
print("Done! Check your ./cache folder now.")





/home/emre/.local/lib/python3.10/site-packages/torchdata/datapipes/__init__.py:18: UserWarning: 
################################################################################
WARNING!
The 'datapipes', 'dataloader2' modules are deprecated and will be removed in a
future torchdata release! Please see https://github.com/pytorch/data/issues/1196
to learn more and leave feedback.
################################################################################

  deprecation_warning()


Porting over and transforming the trainset.


IntProgress(value=0, max=5562)

Porting over and transforming the testset.


IntProgress(value=0, max=695)

Porting over and transforming the valset.


IntProgress(value=0, max=696)

Elapsed time: 3 minutes, 6 seconds, 67 milliseconds
Baking the cache to disk... (this creates the files)
Done! Check your ./cache folder now.
